# 314 — GA-DPO trên Colab L4 (nhánh thứ hai của file 314)
Runtime: **L4** (cần bf16; T4 không chạy được). Chỉ lên A100 nếu D-4 tràn bộ nhớ.
Cặp DPO đã dựng ở máy nhà từ đúng bộ mẫu của notebook R (`raft-r-314`), nằm trong dataset `uyenle0712/gadpo-314-cap`.
Chạy lần lượt D-1 → D-6. Không bấm Stop khi D-4 đang chạy.

In [ ]:
# D-1: gói + kiểm GPU
!pip -q install 'transformers==5.18.0' 'peft==0.21.1' 'kaggle==2.2.4' accelerate safetensors
!pip -q uninstall -y torchao
import torch
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 2**30, "GiB")
assert torch.cuda.get_device_capability()[0] >= 8, "⛔ cần L4/A100 (bf16)"


In [ ]:
# D-1b: gắn Drive NGAY TỪ ĐẦU (lúc bạn còn ngồi máy để bấm cho phép), kết quả sẽ chép vào MyDrive/thesis/gadpo314/
from google.colab import drive
import os
drive.mount("/content/drive")
DR = "/content/drive/MyDrive/thesis/gadpo314"
os.makedirs(DR, exist_ok=True)
print("[drive]", DR, sorted(os.listdir(DR)), flush=True)


In [ ]:
# D-2: khoá Kaggle (chọn tệp ~/.kaggle/uyenle/access_token) + tải dữ liệu
import os
from google.colab import files
up = files.upload()
ten = next(iter(up))
os.environ["KAGGLE_API_TOKEN"] = open(ten).read().strip()   # không in khoá
os.remove(ten)
for ds, d in [("uyenle0712/gadpo-314", "/content/code"),
              ("uyenle0712/gadpo-314-cap", "/content/cap"),
              ("uyenle0712/fgrb-p1-bundle", "/content/bundle"),
              ("uyenle0712/grpo-spice-ck500", "/content/ck500")]:
    r = os.system(f"kaggle datasets download -d {ds} -p {d} --unzip -q")
    print(ds, "→", d, "OK" if r == 0 else f"LỖI {r}", flush=True)
    assert r == 0


In [ ]:
# D-3: đường dẫn, chép mã, hoà S1 ở fp16 (khớp đường chấm Kaggle T4), đổi đường dẫn ảnh
import glob, shutil, subprocess, json, time
W = "/content/w"; os.makedirs(W, exist_ok=True)
M = "/content/s1_merged"
BUNDLE = next(r for r, d, f in os.walk("/content/bundle") if "adapter_s1_seed101" in d and "images" in d)
CK = next(r for r, d, f in os.walk("/content/ck500") if "adapter_config.json" in f)
CODE = os.path.dirname(next(glob.iglob("/content/code/**/ga_dpo.py", recursive=True)))
CAP = next(glob.iglob("/content/cap/**/cap_all.jsonl", recursive=True))
for f in os.listdir(CODE):
    if f.endswith(".py"): shutil.copy(f"{CODE}/{f}", W)
print("BUNDLE", BUNDLE, "\nCK", CK, "\nCAP", CAP, flush=True)
ENV = {**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1",
       "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
def run(*a):
    r = subprocess.run(["python", *a], cwd=W, capture_output=True, text=True, env=ENV)
    print((r.stdout + r.stderr)[-3000:], flush=True); assert r.returncode == 0
run("-c", "import sys, argparse, torch, grpo_spice as G; G._dtype = lambda: torch.float16; "
          "G.merge(argparse.Namespace(bundle=sys.argv[1], merged=sys.argv[2], adapter=sys.argv[1] + '/adapter_s1_seed101'))",
    BUNDLE, M)
run("mau_sang_ga.py", "--vao", CAP, "--ra", f"{W}/cap_colab.jsonl", "--anh-moi", f"{BUNDLE}/images")
P = [json.loads(l) for l in open(f"{W}/cap_colab.jsonl")]
miss = [p["image"] for p in P if not os.path.exists(p["image"])]
print("cặp:", len(P), "· cải thiện", sum(p["loai"] == "cai_thien" for p in P), "· giữ", sum(p["loai"] == "giu" for p in P), "· thiếu ảnh:", len(miss))
assert not miss


In [ ]:
# D-4: train GA-DPO (DPO + NLL, β 0,1, λ 1, lr 5e-6, tích luỹ 16 cặp, 1 lượt) — chạy nền, in tiến độ mỗi 2 phút
T0 = time.time()
log = "/content/train_dpo.log"
p = subprocess.Popen(["python", "ga_dpo.py", "--train", "--merged", M, "--ck500", CK,
                      "--cap-file", f"{W}/cap_colab.jsonl", "--out", "/content/dpo", "--kieu", "dpo"],
                     cwd=W, env=ENV, stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True)
while p.poll() is None:
    time.sleep(120)
    dong = [l for l in open(log).read().splitlines() if l.strip()]
    print(time.strftime("%H:%M"), f"{(time.time()-T0)/60:.0f} phút",
          f"VRAM ~{torch.cuda.mem_get_info()[1]/2**30 - torch.cuda.mem_get_info()[0]/2**30:.1f} GiB đang dùng",
          dong[-1:] if dong else "", flush=True)
    shutil.copy(log, f"{DR}/train_dpo_dangchay.log")  # log sống sót nếu mất máy
print(open(log).read()[-3000:])
assert p.returncode == 0, "⛔ train lỗi — xem log ở trên (OOM thì đổi runtime A100 rồi chạy lại từ D-1)"


In [ ]:
# D-5: ghép adapter DPO với ck500 thành MỘT adapter trên S1 hoà (ghép hạng, chính xác)
cfg_ck = json.load(open(f"{CK}/adapter_config.json"))
pc = "/content/dpo/final/adapter_config.json"
c = json.load(open(pc)); c["base_model_name_or_path"] = cfg_ck.get("base_model_name_or_path")
json.dump(c, open(pc, "w"), indent=2)
run("tron_adapter.py", "--vao", f"{CK}:1", "--vao", "/content/dpo/final:1", "--ra", "/content/ck500_dpo")
shutil.copy(log, "/content/ck500_dpo/train_dpo.log")
print(sorted(os.listdir("/content/ck500_dpo")))


In [ ]:
# D-5b: chép adapter (DPO riêng + bản ghép với ck500) và log lên Drive — phòng D-6 lỗi hoặc mất máy sau đó
import shutil, os, hashlib
for nguon, dich in [("/content/dpo/final", f"{DR}/dpo_final"), ("/content/ck500_dpo", f"{DR}/ck500_dpo")]:
    shutil.copytree(nguon, dich, dirs_exist_ok=True)
    for t in sorted(os.listdir(nguon)):
        a, b = os.path.join(nguon, t), os.path.join(dich, t)
        if os.path.isfile(a):
            assert os.path.getsize(a) == os.path.getsize(b), f"⛔ chép lệch {t}"
    print("[drive] đã chép", nguon, "→", dich, sorted(os.listdir(dich)), flush=True)
shutil.copy(log, f"{DR}/train_dpo.log")
print("md5 adapter ghép:", hashlib.md5(open(f"{DR}/ck500_dpo/adapter_model.safetensors", "rb").read()).hexdigest(), flush=True)


In [ ]:
# D-6: đẩy adapter ghép lên Kaggle (dataset riêng tư) để notebook E chấm
json.dump({"title": "gadpo-314-adapter", "id": "uyenle0712/gadpo-314-adapter", "licenses": [{"name": "CC0-1.0"}]},
          open("/content/ck500_dpo/dataset-metadata.json", "w"))
r = os.system("kaggle datasets create -p /content/ck500_dpo -q")
print("đẩy:", "OK" if r == 0 else f"LỖI {r}")


## Lượt 2 — lr 5e-5 (lượt 1 với lr 5e-6 không dịch được mô hình: marg ≈ 0 suốt 52 bước)
Chạy tiếp trong CÙNG phiên (không chạy lại D-1…D-3). Mất máy thì chạy D-1 → D-3 rồi nhảy thẳng xuống D-7.
Lượt 1 giữ nguyên trên Drive (`gadpo314/`) và Kaggle (`gadpo-314-adapter`); lượt 2 ghi sang chỗ riêng.

In [ ]:
# D-7: train lại GA-DPO với lr 5e-5 (mọi thứ khác như D-4) — chạy nền, in tiến độ mỗi 2 phút
LR2 = "5e-5"; OUT2 = "/content/dpo_lr5e5"; DR2 = f"{DR}/lr5e5"; os.makedirs(DR2, exist_ok=True)
assert not os.path.exists(OUT2), f"⛔ {OUT2} đã có — xoá đi hoặc đổi tên trước khi chạy"
T0 = time.time()
log2 = "/content/train_dpo_lr5e5.log"
p = subprocess.Popen(["python", "ga_dpo.py", "--train", "--merged", M, "--ck500", CK, "--lr", LR2,
                      "--cap-file", f"{W}/cap_colab.jsonl", "--out", OUT2, "--kieu", "dpo"],
                     cwd=W, env=ENV, stdout=open(log2, "w"), stderr=subprocess.STDOUT, start_new_session=True)
while p.poll() is None:
    time.sleep(120)
    dong = [l for l in open(log2).read().splitlines() if l.strip()]
    print(time.strftime("%H:%M"), f"{(time.time()-T0)/60:.0f} phút", dong[-1:] if dong else "", flush=True)
    shutil.copy(log2, f"{DR2}/train_dpo_dangchay.log")
print(open(log2).read()[-3000:])
assert p.returncode == 0, "⛔ train lỗi — xem log ở trên"
assert f"lr {float(LR2)}" in open(log2).read() or f"lr {LR2}" in open(log2).read(), "⛔ log không ghi lr mới — kiểm dòng [train]"

In [ ]:
# D-8: ghép với ck500 + chép lên Drive (gadpo314/lr5e5/)
import hashlib
pc = f"{OUT2}/final/adapter_config.json"
c = json.load(open(pc)); c["base_model_name_or_path"] = cfg_ck.get("base_model_name_or_path")
json.dump(c, open(pc, "w"), indent=2)
GHEP2 = "/content/ck500_dpo_lr5e5"
run("tron_adapter.py", "--vao", f"{CK}:1", "--vao", f"{OUT2}/final:1", "--ra", GHEP2)
shutil.copy(log2, f"{GHEP2}/train_dpo.log")
for nguon, dich in [(f"{OUT2}/final", f"{DR2}/dpo_final"), (GHEP2, f"{DR2}/ck500_dpo")]:
    shutil.copytree(nguon, dich, dirs_exist_ok=True)
    for t in sorted(os.listdir(nguon)):
        if os.path.isfile(f"{nguon}/{t}"):
            assert os.path.getsize(f"{nguon}/{t}") == os.path.getsize(f"{dich}/{t}"), f"⛔ chép lệch {t}"
    print("[drive]", nguon, "→", dich, sorted(os.listdir(dich)), flush=True)
shutil.copy(log2, f"{DR2}/train_dpo.log")
print("md5 adapter ghép lượt 2:", hashlib.md5(open(f"{DR2}/ck500_dpo/adapter_model.safetensors", "rb").read()).hexdigest(), flush=True)

In [ ]:
# D-9: đẩy adapter ghép lượt 2 lên Kaggle (dataset riêng tư, tên khác lượt 1)
json.dump({"title": "gadpo-314-adapter-lr5e5", "id": "uyenle0712/gadpo-314-adapter-lr5e5", "licenses": [{"name": "CC0-1.0"}]},
          open(f"{GHEP2}/dataset-metadata.json", "w"))
r = os.system(f"kaggle datasets create -p {GHEP2} -q")
print("đẩy:", "OK" if r == 0 else f"LỖI {r}")